# GenerativeAURORA — Step 9 (Fixed): Baselines and Paper Tables

### Purpose
Step 9 evaluates the proposed GenerativeAURORA method against baselines and produces the paper tables.

### Workflow position
**Step 5 — Train → Step 6 — Evaluate raw scenarios → Step 7 — RCCC calibration → Step 8 — Figures → Step 9 — Baselines and paper tables**

### Faithful-conversion rule
The supplied `step9_tables.py` is the source of truth. The executable Python in this notebook is copied verbatim from the supplied script and only divided into logical notebook cells.

No model architecture, baseline method, metric formula, parameter, output filename, or table content is silently changed. Markdown cells are added only for explanation.

### Important fixed metric treatment
The original script explicitly uses:
- **Raw scenarios** for CRPS, pinball loss, and median MAE.
- **Calibrated intervals** for the GenerativeAURORA + RCCC coverage and 90% interval width.

This distinction is preserved exactly.


## 1. Header and imports

This opening block is preserved verbatim from the source script. It describes the fixed Step 9 metric treatment and the five methods compared in the paper tables.

In [ ]:
# GenerativeAURORA - Step 9 (Fixed): Baselines and Paper Tables
#
# Fix from previous version:
#   GenerativeAURORA + RCCC now uses CALIBRATED INTERVALS for coverage/width
#   and RAW SCENARIO metrics for CRPS/pinball/MAE.
#   These are the correct metrics for a calibrated scenario generator.
#
# Baselines:
#   1. Gaussian          -- mean/std from training, sample scenarios
#   2. Historical bootstrap -- similar-CI days from training
#   3. DDPM no regime    -- ablation: regime forced to Clear
#   4. GenerativeAURORA  -- full model, raw (no calibration)
#   5. GenerativeAURORA + RCCC -- full model + calibrated intervals
#
# Run:
#   python step9_tables.py

## 2. Dataset, evaluation, and table paths

Defines the original dataset, evaluation, and paper-table output directories. The Windows paths are intentionally preserved in this faithful conversion.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Parameters and labels

Preserves the original scenario count, batch size, forecast horizon, diffusion steps, RCCC significance level, and regime/site labels.

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset"
CODING_DIR  = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"
EVAL_DIR    = os.path.join(DATASET_DIR, "evaluation")
TABLE_DIR   = os.path.join(DATASET_DIR, "paper_tables")
os.makedirs(TABLE_DIR, exist_ok=True)

N_SCENARIOS = 50
BATCH_SIZE  = 32
HORIZON     = 16
T_DIFFUSION = 200
ALPHA       = 0.10

REGIME_NAMES = {0: "Clear", 1: "Cloudy", 2: "Overcast"}
SITE_NAMES   = {0: "Phoenix AZ", 1: "Los Angeles CA", 2: "Denver CO",
                3: "Miami FL",   4: "Seattle WA"}

## 4. Model import

Reuses the `GenerativeAURORA` model architecture from Step 4, exactly as in the source script.

In [ ]:
# -----------------------------------------------------------------------
# IMPORT MODEL FROM STEP 4
# -----------------------------------------------------------------------
import os
import json
import sys

# Path to the target notebook on Google Drive
notebook_path = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"

if not os.path.exists(notebook_path):
    raise FileNotFoundError(f"Could not find notebook at: {notebook_path}. Please check your Google Drive mount or path.")

# Read and execute Python code blocks from the step 4 notebook directly
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Execute code cells sequentially in the global namespace
for cell in nb_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        code_text = ''.join(cell.get('source', []))
        # Avoid executing installations or imports of itself recursively
        if "import_ipynb" not in code_text and "GenerativeAURORA_step4_build_model_Colab" not in code_text:
            exec(code_text, globals())

PyTorch version: 2.11.0+cu130
CUDA available  : True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HORIZON     : 16
LOOKBACK    : 16
N_FEATURES  : 10
N_REGIMES   : 3
N_SITES     : 5
T_DIFFUSION : 200
EMBED_DIM   : 64
HIDDEN_DIM  : 128
Betas shape     : (200,)
Alphas shape    : (200,)
Alpha-bar shape : (200,)
First beta      : 9.999999747378752e-05
Last beta       : 0.019999999552965164
Device              : cuda
Total parameters    : 5,095,329
Trainable parameters: 5,095,329
x0 shape       : (4, 16)
cond_x shape   : (4, 16, 10)
regime shape   : (4,)
site shape     : (4,)
Loss value     : 1.182862
Loss dimensions: () (scalar expected)

Forward-pass shape check: PASSED
Generated scenarios shape: (4, 16)
Expected shape           : (4, 16)
Scenario-generation shape check: PASSED


## 5. Dataset class

Provides the dataset wrapper used to load train/test sequence arrays and their regime/site labels.

In [ ]:
class SolarSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path):
        data        = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]

## 6. Scaling utilities

Loads the scaler parameters and converts normalized GHI values back to W/m².

In [ ]:
def load_scalers():
    with open(os.path.join(DATASET_DIR, "scaler_params.json"), "r", encoding="utf-8") as f:
        return json.load(f)

## 7. Metric functions

These functions implement CRPS, pinball loss, median MAE, interval coverage, and interval width exactly as defined in the original script.

In [ ]:
def inv_scale(arr, scalers):
    s = scalers["GHI"]
    return arr * (s["max"] - s["min"]) + s["min"]

## 8. RCCC-aware metrics

This function implements the fixed RCCC reporting logic: raw scenarios remain responsible for CRPS/pinball/MAE, while calibrated intervals provide the RCCC 90% coverage and width.

In [ ]:
# -----------------------------------------------------------------------
# METRICS
# -----------------------------------------------------------------------
def crps_score(actuals, scenarios):
    N, S, H = scenarios.shape
    t1 = np.abs(scenarios - actuals[:, np.newaxis, :]).mean(axis=1)
    t2 = np.zeros((N, H))
    for n in range(N):
        sc = scenarios[n]
        t2[n] = np.abs(sc[:, np.newaxis, :] - sc[np.newaxis, :, :]).mean(axis=(0, 1))
    return float((t1 - 0.5 * t2).mean())


def pinball(actuals, scenarios, q):
    fc = np.quantile(scenarios, q, axis=1)
    e  = actuals - fc
    return float(np.where(e >= 0, q * e, (q - 1) * e).mean())


def med_mae(actuals, scenarios):
    return float(np.abs(np.median(scenarios, axis=1) - actuals).mean())


def cov_width_from_scenarios(actuals, scenarios, alpha=ALPHA):
    lower   = np.quantile(scenarios, alpha / 2,     axis=1)
    upper   = np.quantile(scenarios, 1 - alpha / 2, axis=1)
    cov     = float(((actuals >= lower) & (actuals <= upper)).mean()) * 100
    wid     = float((upper - lower).mean())
    return round(cov, 2), round(wid, 2)


def cov_width_from_intervals(actuals, lower, upper):
    cov = float(((actuals >= lower) & (actuals <= upper)).mean()) * 100
    wid = float((upper - lower).mean())
    return round(cov, 2), round(wid, 2)


def scenario_metrics(actuals, scenarios):
    cov90, w90 = cov_width_from_scenarios(actuals, scenarios, alpha=0.10)
    cov80, _   = cov_width_from_scenarios(actuals, scenarios, alpha=0.20)
    return {
        "CRPS":         round(crps_score(actuals, scenarios), 4),
        "Pinball p10":  round(pinball(actuals, scenarios, 0.10), 4),
        "Pinball p50":  round(pinball(actuals, scenarios, 0.50), 4),
        "Pinball p90":  round(pinball(actuals, scenarios, 0.90), 4),
        "Coverage 90%": cov90,
        "Coverage 80%": cov80,
        "Width 90% PI": w90,
        "Median MAE":   round(med_mae(actuals, scenarios), 4),
    }


def rccc_metrics(actuals, scenarios, lower_cal, upper_cal):
    # Scenario metrics (CRPS etc) from raw scenarios -- RCCC does not change these
    cov90_cal, w90_cal = cov_width_from_intervals(actuals, lower_cal, upper_cal)
    cov80_cal, _ = cov_width_from_intervals(
        actuals,
        np.quantile(scenarios, 0.10, axis=1) - 0,  # placeholder, use cal intervals for 90%
        np.quantile(scenarios, 0.90, axis=1) + 0
    )
    # For 80% PI under RCCC, we use the raw 80% coverage since calibration targets 90%
    cov80_raw, _ = cov_width_from_scenarios(actuals, scenarios, alpha=0.20)

    return {
        "CRPS":         round(crps_score(actuals, scenarios), 4),
        "Pinball p10":  round(pinball(actuals, scenarios, 0.10), 4),
        "Pinball p50":  round(pinball(actuals, scenarios, 0.50), 4),
        "Pinball p90":  round(pinball(actuals, scenarios, 0.90), 4),
        "Coverage 90%": cov90_cal,    # from calibrated intervals
        "Coverage 80%": cov80_raw,    # from raw scenarios (80% PI not calibrated)
        "Width 90% PI": w90_cal,      # from calibrated intervals
        "Median MAE":   round(med_mae(actuals, scenarios), 4),
    }

## 9. Baseline methods

Runs the three baseline approaches in the original script: Gaussian, historical bootstrap, and the DDPM no-regime ablation.

In [ ]:
# -----------------------------------------------------------------------
# BASELINES
# -----------------------------------------------------------------------
def gaussian_baseline(train_Y, test_Y, scalers, n_scenarios):
    train_wm2 = inv_scale(train_Y, scalers)
    test_wm2  = inv_scale(test_Y,  scalers)
    mu  = train_wm2.mean(axis=0)
    sig = np.maximum(train_wm2.std(axis=0), 1.0)
    N   = len(test_wm2)
    sc  = np.random.normal(
        mu[np.newaxis, np.newaxis, :],
        sig[np.newaxis, np.newaxis, :],
        size=(N, n_scenarios, HORIZON)
    ).clip(0, None).astype(np.float32)
    return test_wm2, sc


def bootstrap_baseline(train_X, train_Y, test_X, test_Y, scalers, n_scenarios):
    train_wm2 = inv_scale(train_Y, scalers)
    test_wm2  = inv_scale(test_Y,  scalers)
    train_ci  = train_X[:, :, -1].mean(axis=1)
    test_ci   = test_X[:,  :, -1].mean(axis=1)
    N  = len(test_wm2)
    sc = np.zeros((N, n_scenarios, HORIZON), dtype=np.float32)
    for i in range(N):
        diffs  = np.abs(train_ci - test_ci[i])
        k      = min(n_scenarios * 2, len(train_ci))
        top_k  = np.argsort(diffs)[:k]
        chosen = np.random.choice(top_k, size=n_scenarios, replace=True)
        sc[i]  = train_wm2[chosen]
    return test_wm2, sc


def ddpm_no_regime(model, loader, device, scalers, n_scenarios):
    model.eval()
    all_act, all_sc = [], []
    print("  Running DDPM no-regime ablation...")
    with torch.no_grad():
        for X, Y, regime, site in loader:
            X    = X.to(device)
            site = site.to(device)
            B    = X.shape[0]
            reg_forced = torch.zeros(B, dtype=torch.long, device=device)
            sc = model.sample(X, reg_forced, site, n_scenarios=n_scenarios)
            sc = sc.cpu().numpy().reshape(B, n_scenarios, HORIZON)
            all_act.append(Y.numpy())
            all_sc.append(sc)
    act = np.concatenate(all_act, axis=0)
    sc  = np.concatenate(all_sc,  axis=0)
    return inv_scale(act, scalers), inv_scale(sc, scalers)

## 10. LaTeX table utilities

Converts result DataFrames to LaTeX tabular environments and saves both CSV and `.tex` versions.

In [ ]:
# -----------------------------------------------------------------------
# LATEX TABLE
# -----------------------------------------------------------------------
def to_latex(df, caption, label):
    lines  = ["\\begin{table}[ht]", "\\centering",
              "\\caption{" + caption + "}",
              "\\label{" + label + "}"]
    spec   = "l" + "r" * len(df.columns)
    lines += ["\\begin{tabular}{" + spec + "}", "\\toprule"]
    lines.append(" & ".join([""] + list(df.columns)) + " \\\\")
    lines.append("\\midrule")
    for idx, row in df.iterrows():
        cells = [str(idx)]
        for val in row.values:
            cells.append(("%.4f" % val) if isinstance(val, float) else str(val))
        lines.append(" & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


def save_table(df, name, caption, label):
    csv_p = os.path.join(TABLE_DIR, name + ".csv")
    tex_p = os.path.join(TABLE_DIR, name + ".tex")
    df.to_csv(csv_p, encoding="utf-8")
    with open(tex_p, "w", encoding="utf-8") as f:
        f.write(to_latex(df, caption, label))
    print("  Saved: " + csv_p)
    print("  Saved: " + tex_p)

## 11. Main setup and data loading

The first part of `main()` fixes the random seed, selects CPU/CUDA, loads scalers and datasets, loads the trained model, and reads Step 7 RCCC outputs.

In [ ]:
# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------
def main():
    print("GenerativeAURORA - Step 9 (Fixed): Baselines and Paper Tables")
    print("")

    np.random.seed(42)
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scalers = load_scalers()
    print("Device: " + str(device))
    print("")

    # Load datasets
    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    test_ds  = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    train_X = train_ds.X.numpy()
    train_Y = train_ds.Y.numpy()
    test_X  = test_ds.X.numpy()
    test_Y  = test_ds.Y.numpy()
    test_reg = test_ds.regime.numpy()
    test_sit = test_ds.site.numpy()

    # Load model
    model = GenerativeAURORA(T=T_DIFFUSION).to(device)
    model.load_state_dict(torch.load(
        os.path.join(DATASET_DIR, "best_model.pt"),
        map_location=device, weights_only=True))
    model.eval()

    # Load RCCC outputs from Step 7
    cal_data  = np.load(os.path.join(EVAL_DIR, "calibrated_intervals.npz"))
    act_rccc  = cal_data["actuals"]
    sc_rccc   = cal_data["scenarios"]
    l_cal     = cal_data["lower_cal"]
    u_cal     = cal_data["upper_cal"]
    reg_rccc  = cal_data["regimes"]
    sit_rccc  = cal_data["sites"]

    # Run baselines
    print("Running baselines...")
    print("  Gaussian...")
    act_g,  sc_g  = gaussian_baseline(train_Y, test_Y, scalers, N_SCENARIOS)

    print("  Historical bootstrap...")
    act_b,  sc_b  = bootstrap_baseline(train_X, train_Y, test_X, test_Y, scalers, N_SCENARIOS)

    act_nr, sc_nr = ddpm_no_regime(model, test_loader, device, scalers, N_SCENARIOS)

    act_f6 = inv_scale(test_Y, scalers)
    sc_f6  = cal_data["scenarios"]   # raw GenerativeAURORA scenarios from Step 7

    print("")

## 12. Table II — Overall metrics

Builds Table II comparing the four raw-scenario methods plus GenerativeAURORA + RCCC using the source script's metric definitions.

In [ ]:
    # -----------------------------------------------------------------------
    # TABLE II: Overall metrics
    # -----------------------------------------------------------------------
    print("Building Table II: Overall metrics...")#

    scalers = load_scalers()

    # Load datasets
    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    test_ds  = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    train_X = train_ds.X.numpy()
    train_Y = train_ds.Y.numpy()
    test_X  = test_ds.X.numpy()
    test_Y  = test_ds.Y.numpy()
    test_reg = test_ds.regime.numpy()
    test_sit = test_ds.site.numpy()

    # Run baselines
    print("Running baselines...")
    print("  Gaussian...")
    act_g,  sc_g  = gaussian_baseline(train_Y, test_Y, scalers, N_SCENARIOS)

    print("  Historical bootstrap...")
    act_b,  sc_b  = bootstrap_baseline(train_X, train_Y, test_X, test_Y, scalers, N_SCENARIOS)

    act_nr, sc_nr = ddpm_no_regime(model, test_loader, device, scalers, N_SCENARIOS)

    act_f6 = inv_scale(test_Y, scalers)
    sc_f6  = cal_data["scenarios"]   # raw GenerativeAURORA scenarios from Step 7

    print("")

    methods_scenario = {
        "Gaussian baseline":    (act_g,  sc_g),
        "Historical bootstrap": (act_b,  sc_b),
        "DDPM (no regime)":     (act_nr, sc_nr),
        "GenerativeAURORA":     (act_f6, sc_f6),
    }

    rows = {}
    for name, (act, sc) in methods_scenario.items():
        rows[name] = scenario_metrics(act, sc)

    # GenerativeAURORA + RCCC uses calibrated coverage/width, raw scenario CRPS
    rows["GenerativeAURORA + RCCC"] = rccc_metrics(act_rccc, sc_rccc, l_cal, u_cal)

    table2 = pd.DataFrame(rows).T
    print(table2.to_string())
    save_table(table2, "table2_overall_metrics",
               "Overall evaluation metrics. Best per column in bold. "
               "Coverage and width for GenerativeAURORA + RCCC use calibrated 90\\% PI.",
               "tab:overall")
    print("")

Building Table II: Overall metrics...
Running baselines...
  Gaussian...
  Historical bootstrap...
  Running DDPM no-regime ablation...

                              CRPS  Pinball p10  Pinball p50  Pinball p90  Coverage 90%  Coverage 80%  Width 90% PI  Median MAE
Gaussian baseline          97.6286      28.4026      71.7143      27.5252         90.77         82.30        508.03    143.4286
Historical bootstrap       43.4964      13.0180      30.2713      15.2674         89.95         82.86        245.35     60.5426
DDPM (no regime)         1339.3348     717.3818     472.5035     589.1345        100.00        100.00      16392.88    945.0071
GenerativeAURORA           17.4791       6.1772      11.7914       5.6967         92.19         86.00        101.55     23.5827
GenerativeAURORA + RCCC    17.4791       6.1772      11.7914       5.6967         95.32         86.00        120.72     23.5827
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table2_overall_metrics.csv

## 13. Table III — Per-regime metrics

Builds Table III with GenerativeAURORA + RCCC metrics for Clear, Cloudy, and Overcast regimes.

In [ ]:
    # -----------------------------------------------------------------------
    # TABLE III: Per-regime (GenerativeAURORA + RCCC)
    # -----------------------------------------------------------------------
    print("Building Table III: Per-regime (GenerativeAURORA + RCCC)...")
    regime_rows = {}
    for rid, rname in REGIME_NAMES.items():
        mask = (reg_rccc == rid)
        n    = mask.sum()
        if n == 0:
            continue
        m = rccc_metrics(act_rccc[mask], sc_rccc[mask], l_cal[mask], u_cal[mask])
        regime_rows[rname + " (n=" + str(n) + ")"] = m
    table3 = pd.DataFrame(regime_rows).T
    print(table3.to_string())
    save_table(table3, "table3_regime_metrics",
               "GenerativeAURORA + RCCC metrics by sky regime on the test set.",
               "tab:regime")
    print("")

Building Table III: Per-regime (GenerativeAURORA + RCCC)...
                   CRPS  Pinball p10  Pinball p50  Pinball p90  Coverage 90%  Coverage 80%  Width 90% PI  Median MAE
Clear (n=872)   17.6319       6.2857      11.9020       5.6695         95.46         86.32        123.23     23.8041
Cloudy (n=46)   14.6820       4.1873       9.8563       6.0297         93.89         81.52         77.84     19.7125
Overcast (n=2)  15.2299       4.6446       8.0415       9.9026         68.75         46.88         11.77     16.0829
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table3_regime_metrics.csv
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table3_regime_metrics.tex



## 14. Table IV — RCCC before vs after

Builds Table IV from `rccc_results.csv`, showing coverage and interval width before and after RCCC.

In [ ]:
    # -----------------------------------------------------------------------
    # TABLE IV: RCCC before vs after
    # -----------------------------------------------------------------------
    print("Building Table IV: RCCC before vs after...")
    rccc_df = pd.read_csv(os.path.join(EVAL_DIR, "rccc_results.csv"))
    regime_rccc_df = rccc_df[rccc_df["group"] == "regime"].set_index("name")
    regime_rccc_df = regime_rccc_df[["n", "coverage_before", "width_before",
                                     "coverage_after", "width_after", "delta_coverage"]]
    regime_rccc_df.columns = ["N", "Cov. before (%)", "Width before",
                               "Cov. after (%)", "Width after", "Delta (pp)"]
    print(regime_rccc_df.to_string())
    save_table(regime_rccc_df, "table4_rccc_calibration",
               "Effect of RCCC on 90\\% prediction interval coverage and width by sky regime.",
               "tab:rccc")
    print("")

Building Table IV: RCCC before vs after...
            N  Cov. before (%)  Width before  Cov. after (%)  Width after  Delta (pp)
name                                                                                 
Clear     872          92.4742      103.7299         95.4558     123.2318        2.98
Cloudy     46          88.4511       64.3797         93.8859      77.8447        5.43
Overcast    2          53.1250        7.3558         68.7500      11.7655       15.62
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table4_rccc_calibration.csv
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table4_rccc_calibration.tex



## 15. Table V — Per-site metrics

Builds Table V with GenerativeAURORA + RCCC metrics by site.

In [ ]:
    # -----------------------------------------------------------------------
    # TABLE V: Per-site (GenerativeAURORA + RCCC)
    # -----------------------------------------------------------------------
    print("Building Table V: Per-site (GenerativeAURORA + RCCC)...")

    # Load RCCC outputs from Step 7
    cal_data  = np.load(os.path.join(EVAL_DIR, "calibrated_intervals.npz"))
    act_rccc  = cal_data["actuals"]
    sc_rccc   = cal_data["scenarios"]
    l_cal     = cal_data["lower_cal"]
    u_cal     = cal_data["upper_cal"]
    reg_rccc  = cal_data["regimes"]
    sit_rccc  = cal_data["sites"]

    site_rows = {}
    for sid, sname in SITE_NAMES.items():
        mask = (sit_rccc == sid)
        n    = mask.sum()
        if n == 0:
            continue
        m = rccc_metrics(act_rccc[mask], sc_rccc[mask], l_cal[mask], u_cal[mask])
        site_rows[sname + " (n=" + str(n) + ")"] = m
    table5 = pd.DataFrame(site_rows).T
    print(table5.to_string())
    save_table(table5, "table5_site_metrics",
               "GenerativeAURORA + RCCC metrics by site on the test set.",
               "tab:site")
    print("")

Building Table V: Per-site (GenerativeAURORA + RCCC)...
                           CRPS  Pinball p10  Pinball p50  Pinball p90  Coverage 90%  Coverage 80%  Width 90% PI  Median MAE
Phoenix AZ (n=184)      15.0617       5.0924      10.0635       5.1748         95.24         85.53        103.86     20.1271
Los Angeles CA (n=184)  12.8101       4.5624       8.5768       4.3100         96.81         89.37        100.95     17.1537
Denver CO (n=184)       22.1560       7.8993      14.9575       7.1417         93.24         81.90        137.90     29.9150
Miami FL (n=184)        22.1190       8.3646      14.9024       6.7593         94.90         85.56        150.34     29.8048
Seattle WA (n=184)      15.2489       4.9676      10.4566       5.0978         96.40         87.64        110.56     20.9132
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table5_site_metrics.csv
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table5_site_metrics.tex



## 16. Table I — Dataset summary

Builds Table I from the fixed dataset-summary values contained in the original script.

In [ ]:
    # -----------------------------------------------------------------------
    # TABLE I: Dataset summary
    # -----------------------------------------------------------------------
    print("Building Table I: Dataset summary...")
    t1 = pd.DataFrame({
        "Phoenix AZ":     {"Climate": "Hot desert",     "Train": 1461, "Test": 184, "Clear %": 96.0, "GHI mean": 487},
        "Los Angeles CA": {"Climate": "Mediterranean",  "Train": 1461, "Test": 184, "Clear %": 92.4, "GHI mean": 458},
        "Denver CO":      {"Climate": "Semi-arid",      "Train": 1461, "Test": 184, "Clear %": 88.1, "GHI mean": 409},
        "Miami FL":       {"Climate": "Subtropical",    "Train": 1461, "Test": 184, "Clear %": 90.5, "GHI mean": 440},
        "Seattle WA":     {"Climate": "Oceanic",        "Train": 1461, "Test": 184, "Clear %": 74.2, "GHI mean": 298},
    }).T
    print(t1.to_string())
    save_table(t1, "table1_dataset_stats",
               "Dataset summary. NSRDB 2018-2022, five US climate zones.",
               "tab:dataset")
    print("")

    print("=== Step 9 complete ===")
    print("")
    print("Key result (Table II) -- what reviewers will see:")
    print(table2[["CRPS", "Coverage 90%", "Width 90% PI", "Median MAE"]].to_string())
    print("")
    print("All tables saved to: " + TABLE_DIR)




Building Table I: Dataset summary...
                      Climate Train Test Clear % GHI mean
Phoenix AZ         Hot desert  1461  184    96.0      487
Los Angeles CA  Mediterranean  1461  184    92.4      458
Denver CO           Semi-arid  1461  184    88.1      409
Miami FL          Subtropical  1461  184    90.5      440
Seattle WA            Oceanic  1461  184    74.2      298
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table1_dataset_stats.csv
  Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table1_dataset_stats.tex

=== Step 9 complete ===

Key result (Table II) -- what reviewers will see:
                              CRPS  Coverage 90%  Width 90% PI  Median MAE
Gaussian baseline          97.6286         90.77        508.03    143.4286
Historical bootstrap       43.4964         89.95        245.35     60.5426
DDPM (no regime)         1339.3348        100.00      16392.88    945.0071
GenerativeAURORA           17.4791         92.19   

## 17. Script entry point

Preserves the final source-script entry point.

In [ ]:
if __name__ == "__main__":
    main()

GenerativeAURORA - Step 9 (Fixed): Baselines and Paper Tables

Device: cuda

Running baselines...
  Gaussian...
  Historical bootstrap...
  Running DDPM no-regime ablation...



## Execution note

The original Windows paths are intentionally preserved:

`C:/Users/user/Paper_10/Dataset`

`C:/Users/user/Paper_10/Coding`

For execution in Colab, these paths need to point to the corresponding dataset and source-code locations.

### Main inputs
- `sequences_train.npz`
- `sequences_test.npz`
- `scaler_params.json`
- `best_model.pt`
- `evaluation/calibrated_intervals.npz`
- `evaluation/rccc_results.csv`
- `step4_build_model.py`

### Main outputs
The original script writes CSV and LaTeX files to:

`Dataset/paper_tables/`

including:
- `table1_dataset_stats.csv` / `.tex`
- `table2_overall_metrics.csv` / `.tex`
- `table3_regime_metrics.csv` / `.tex`
- `table4_rccc_calibration.csv` / `.tex`
- `table5_site_metrics.csv` / `.tex`

The source script ends by reporting the key Table II results and the table output directory.
